In [ ]:
import os

#El modelo y la caché de HF se quedan en el volumen, así no se descarga dos veces
os.environ["HF_HOME"] = "/workspace/hf"

#El token va como variable de entorno del pod, no aquí (el notebook acaba en el repo)
print("HF_TOKEN configurado:", "HF_TOKEN" in os.environ)

In [ ]:
%cd /workspace
!git clone https://github.com/vllm-project/speculators.git
%cd /workspace/speculators

In [ ]:
# Speculators, data prep and training
!pip install -q uv
!uv venv speculators --python 3.12
!uv pip install --python speculators/bin/python -e .

In [ ]:
# vLLM, la misma versión que me funcionó en Colab
!uv venv vllm --python 3.12
!uv pip install --python vllm/bin/python "vllm==0.33.0"

#FlashInfer kernels are disabled fix (lo de ninja)
!./vllm/bin/vllm download-kernels

In [ ]:
#Apunto las versiones para el repo
!uv pip show --python speculators/bin/python speculators | grep Version
!./vllm/bin/vllm --version

In [ ]:
#Descargo el modelo antes de lanzar vLLM (con HF_TOKEN va bastante más rápido)
!./vllm/bin/hf download BSC-LT/salamandra-7b-instruct-2606

In [ ]:
import json

ruta = "/workspace/data/alia_salamandra_20k_sin_cortadas.jsonl"
print("Tamaño en el pod:", round(os.path.getsize(ruta) / 1e6, 1), "MB  (en Drive: 328 MB)")

#Compruebo que la subida terminó y que todas las líneas son JSON válido
buenas, malas = 0, []
with open(ruta, encoding="utf-8") as archivo:
    for numero, linea in enumerate(archivo, 1):
        try:
            json.loads(linea)
            buenas += 1
        except Exception:
            malas.append(numero)

print("Líneas válidas:", buenas, "(deberían ser 30973)")
print("Líneas rotas:", len(malas), malas[:10])

In [ ]:
#4000 y no 4096: con muestras de 4096 exactos el entrenamiento pide max_tokens=0 a vLLM y falla
#--no-skip-token-freq: hace falta token_freq.pt para el vocabulario reducido de 32000 del draft
!rm -rf /workspace/prepared
!./speculators/bin/speculators prepare-data --model BSC-LT/salamandra-7b-instruct-2606 --data /workspace/data/alia_salamandra_20k_sin_cortadas.jsonl --output /workspace/prepared --seq-length 4000 --no-skip-token-freq
!ls /workspace/prepared

In [ ]:
%pip install -q datasets

In [ ]:
from datasets import load_from_disk

preparado = load_from_disk("/workspace/prepared")
longitudes = [len(x) for x in preparado["input_ids"]]

print("Registros:", len(preparado))
print("Tokens por epoch:", round(sum(longitudes) / 1e6, 1), "M")
print("Máximo:", max(longitudes), "(tiene que ser 4000 o menos)")

In [ ]:
import subprocess

#vLLM en la GPU 0, sacando los hidden states que le pide el entrenamiento
#Solo 35% de la memoria (pesos + caché), el resto queda para el entrenamiento
#8192 para que siempre haya margen por encima de la longitud de entrenamiento
entorno_vllm = {
    **os.environ,
    "CUDA_VISIBLE_DEVICES": "0",
    "PATH": "/workspace/speculators/vllm/bin:" + os.environ["PATH"]
}

with open("/workspace/vllm.log", "w") as archivo:
    servidor = subprocess.Popen(
        [
            "./vllm/bin/python", "scripts/launch_vllm.py",
            "BSC-LT/salamandra-7b-instruct-2606",
            "--",
            "--max-model-len", "8192",
            "--port", "8000",
            "--gpu-memory-utilization", "0.35"
        ],
        stdout=archivo,
        stderr=subprocess.STDOUT,
        env=entorno_vllm,
        start_new_session=True
    )

print("vLLM iniciado:", servidor.pid)

In [ ]:
import time, urllib.request

#Espero a que vLLM responda (la primera vez tarda unos minutos compilando)
while True:
    if servidor.poll() is not None:
        print("vLLM se ha parado, mirar el log:")
        !tail -n 30 /workspace/vllm.log
        break
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/v1/models")
        print("vLLM listo")
        break
    except Exception:
        time.sleep(15)

In [ ]:
#Entrenamiento en la misma GPU 0, en segundo plano para que no muera si se cae el navegador
#Si da OOM: bajar --total-seq-len a 2048 (y --seq-length de prepare-data por debajo)
entorno_train = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}

!rm -rf /workspace/checkpoints

with open("/workspace/train.log", "w") as archivo:
    entrenamiento = subprocess.Popen(
        [
            "./speculators/bin/speculators", "train",
            "--verifier-name-or-path", "BSC-LT/salamandra-7b-instruct-2606",
            "--data-path", "/workspace/prepared",
            "--save-path", "/workspace/checkpoints",
            "--draft-vocab-size", "32000",
            "--epochs", "3",
            "--total-seq-len", "4096",
            "--vllm-endpoint", "http://localhost:8000/v1",
            "--on-missing", "generate"
        ],
        stdout=archivo,
        stderr=subprocess.STDOUT,
        env=entorno_train,
        start_new_session=True
    )

print("Entrenamiento iniciado:", entrenamiento.pid)

In [ ]:
#Ejecutar cuando quiera ver cómo va (no afecta al entrenamiento, solo lee el log)
#Una sola GPU: memoria usada ~25 GB de vLLM + lo del entrenamiento (~70 GB en total)
!tail -n 15 /workspace/train.log
!nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv
!ls /workspace/checkpoints 2>/dev/null

#Tiene que seguir saliendo 0
!grep -c "max_tokens must be" /workspace/train.log

In [ ]:
#Validación de cada epoch: full_acc_0/1/2 = acertar seguidos el 1er, 2º y 3er token propuesto
!grep -A 12 "Validation epoch .* completed" /workspace/train.log | grep -E "full_acc|cond_acc|loss_epoch"

In [ ]:
#None = sigue corriendo, 0 = terminó bien, otro número = error
print("Estado del entrenamiento:", entrenamiento.poll())

#Por si se corta a mitad: qué opción tiene mi versión para seguir desde el último epoch
!./speculators/bin/speculators train --help | grep -i -A2 -E "resume|from-pretrained"

In [ ]:
import signal

#Cuando termine el entrenamiento, paro vLLM
print("Código de salida del entrenamiento:", entrenamiento.poll())

if entrenamiento.poll() is not None:
    os.killpg(servidor.pid, signal.SIGTERM)
    print("vLLM parado")

In [ ]:
#Subo un checkpoint y el log a HF (privado): 0, 1, 2 al acabar cada epoch, y checkpoint_best al final
CHECKPOINT = "checkpoint_best"

!cp /workspace/train.log /workspace/checkpoints/{CHECKPOINT}/
!./vllm/bin/hf upload user/salamandra-7b-eagle3-alia /workspace/checkpoints/{CHECKPOINT} --private